In [24]:
# model libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# image reading libraries
import nibabel as nib
from nilearn import plotting as npl

import nitools as nt
from smarts_cerebellum import regression
from image_processing import overall_image
from image_processing import mirror_lesion
from pipelines import MNISym_coreg_regression
from image_processing import tissue_extractor as te
import smarts_cerebellum.globals as gl

from image_processing import mirror_lesion

from pathlib import Path
import os

In [25]:
# directories
p_df = pd.read_csv('/cifs/diedrichsen/data/smarts_cerebellum/participants_anat.tsv', sep = '\t')

In [26]:
search_path = os.path.join(gl.baseDir, 'Regression')

# files to include in summarized dataframe


In [27]:
suffixes = [
    'MNISym_CSF_coreg_reslice_slope.nii.gz',
    'MNISym_WM_coreg_reslice_slope.nii.gz',
    'MNISym_GM_coreg_reslice_slope.nii.gz',
    'MNISym_T1_coreg_reslice_slope.nii.gz'
]

In [28]:
left_lesion_df = p_df[p_df.LesionSide == 'left ']

In [29]:
for subj in left_lesion_df.subj_id.unique():
    flip_csf = f'{search_path}/{subj}/{subj}_MNISym_CSF_coreg_reslice_slope.nii.gz'
    if not Path(flip_csf).is_file():
        print(f'Skip {subj}')
        continue
    flipped_csf = mirror_lesion.FlipLR(flip_csf)
    nib.save(flipped_csf, f'{search_path}/{subj}/{subj}_MNISym_CSF_coreg_reslice_slope_FlipLR.nii.gz')


Skip CU_2697
Skip JHU_2374


In [30]:
type = 'WM'
for subj in left_lesion_df.subj_id.unique():
    flip = f'{search_path}/{subj}/{subj}_MNISym_{type}_coreg_reslice_slope.nii.gz'
    if not Path(flip).is_file():
        print(f'Skip {subj}')
        continue
    flipped = mirror_lesion.FlipLR(flip)
    nib.save(flipped, f'{search_path}/{subj}/{subj}_MNISym_{type}_coreg_reslice_slope_FlipLR.nii.gz')
    

Skip CU_2697
Skip JHU_2374


In [31]:
type = 'T1'
for subj in left_lesion_df.subj_id.unique():
    flip = f'{search_path}/{subj}/{subj}_MNISym_{type}_coreg_reslice_slope.nii.gz'
    if not Path(flip).is_file():
        print(f'Skip {subj}')
        continue
    flipped = mirror_lesion.FlipLR(flip)
    nib.save(flipped, f'{search_path}/{subj}/{subj}_MNISym_{type}_coreg_reslice_slope_FlipLR.nii.gz')


Skip CU_2697
Skip JHU_2374


In [32]:
type = 'GM'
for subj in left_lesion_df.subj_id.unique():
    flip = f'{search_path}/{subj}/{subj}_MNISym_{type}_coreg_reslice_slope.nii.gz'
    if not Path(flip).is_file():
        print(f'Skip {subj}')
        continue
    flipped = mirror_lesion.FlipLR(flip)
    nib.save(flipped, f'{search_path}/{subj}/{subj}_MNISym_{type}_coreg_reslice_slope_FlipLR.nii.gz')


Skip CU_2697
Skip JHU_2374


In [33]:
from image_processing import utils
import SUITPy as suit


suffixes = [
    'MNISym_CSF_coreg_reslice_slope.nii.gz',
    'MNISym_WM_coreg_reslice_slope.nii.gz',
    'MNISym_GM_coreg_reslice_slope.nii.gz',
    'MNISym_T1_coreg_reslice_slope.nii.gz'
]
flipped_suffixes = [
    'MNISym_CSF_coreg_reslice_slope_FlipLR.nii.gz',
    'MNISym_WM_coreg_reslice_slope_FlipLR.nii.gz',
    'MNISym_GM_coreg_reslice_slope_FlipLR.nii.gz',
    'MNISym_T1_coreg_reslice_slope_FlipLR.nii.gz'
]

def descriptive_dataframe(atlas_df, p_df, subj):
    """
    Creates a descriptive dataframe
    UNIQUE SUBJECT IMAGE, NOT WEEKS! --> weeks under construction!

    Inputs:
        atlas_df (Pandas dataframe): dataframe from atlas summary
        p_info (Pandas dataframe): dataframe with descriptive information for participants

    Outputs:
        atlas_df (Pandas dataframe): updated dataframe (with descriptive information)
    """

    # try doing this for only one subject; then, we can loop through in the call
    refT1 = p_df[p_df.subj_id == subj]['RefT1'].iloc[0].strip()
    subj_df = p_df[(p_df.subj_id == subj) & (p_df.Week.str.strip() == refT1)]


    # mask the row to which we are adding data
    row_mask = (atlas_df.subj_id == subj)
    print(f'Writing data for {subj}')

    atlas_df.loc[row_mask, 'ID'] = subj_df['ID'].values[0]
    atlas_df.loc[row_mask, 'Centre'] = subj_df['Centre'].values[0]
    atlas_df.loc[row_mask, 'RefT1'] = refT1
    atlas_df.loc[row_mask, 'age'] = subj_df['age'].values[0]
    atlas_df.loc[row_mask, 'Gender'] = subj_df.Gender.values[0]
    atlas_df.loc[row_mask, 'isPatient'] = subj_df.isPatient.values[0]
    atlas_df.loc[row_mask, 'LesionSide'] = subj_df.LesionSide.values[0]
    atlas_df.loc[row_mask, 'LesionLocation'] = subj_df.LesionLocation.values[0]
    atlas_df.loc[row_mask, 'handedness'] = subj_df.handedness.values[0]

 
    return atlas_df


# Make summarized dataframe
def make_summarized_dataframe(p_df,
                              search_path,
                              the_atlas, maps, space,
                              ):
    """
    Make full summarized dataframe that has: ROIs for each subject, along with descriptive information

    Inputs:
        p_df (Pandas dataframe): info file for participants
        search_path (str): directory where files are stored (parent directory for all subjects)
        suffixes (tuple of str): suffixes for all files you want to find

        the_atlas (str): cerebellar atlas --> see SUITPy
        maps (str): map to use in summarizing (cerebellar map) --> see SUITPy
        space: space of the files


    Outputs:

    """

    dfs = []

    # loop through all subjects - perform each operation on each subject
    for subj in p_df.subj_id.unique():
        # find their files - returns string list of files
        if subj in left_lesion_df.subj_id.unique():
            file_list = utils.file_search(search_path = search_path, subj_id = subj, suffixes = flipped_suffixes)
        else:
            file_list = utils.file_search(search_path = search_path, subj_id = subj, suffixes = suffixes)


        
        if not file_list:
            continue # skip subjects without the files
        
        # summarize volume in each ROI for each file type
        
        df = suit.summarize_data(
                                 images = file_list,
                                 atlas = the_atlas,
                                 maps = maps,
                                 space = space,
                                 stats = ['mean', 'median', 'nansum']

        )
        
        df['subj_id']= subj

        # then make the descriptive dataframe for each subject
        descriptive_df = descriptive_dataframe(atlas_df = df, p_df = p_df, subj = subj)

        # add all dataframes to the list
        dfs.append(descriptive_df)

    # combine all of them
    all_df = pd.concat(dfs, ignore_index = True)

    

    return all_df

In [34]:
summarized_df = make_summarized_dataframe(p_df = p_df,
                                             search_path = search_path,
                                
                                             the_atlas = 'Diedrichsen_2009',
                                             maps = 'atl-Anatom',
                                             space = 'MNISym'
                                             )


Writing data for CU_2310
Writing data for CU_2538
Writing data for CU_2663
Writing data for CU_2925
Writing data for JHU_2282
Writing data for JHU_2395
Writing data for JHU_2531
Writing data for JHU_2577
Writing data for JHU_2650
Writing data for JHU_2684
Writing data for JHU_2713
Writing data for JHU_2789
Writing data for JHU_3175
Writing data for JHU_3176
Writing data for UZ_2365
Writing data for UZ_2450
Writing data for UZ_2565
Writing data for UZ_2595
Writing data for UZ_2652
Writing data for UZ_2654
Writing data for UZ_2906
Writing data for UZ_3030
Writing data for UZ_3057
Writing data for UZ_3151
Writing data for UZ_3158
Writing data for UZ_3166
Writing data for UZ_3224
Writing data for UZ_3226
Writing data for UZ_3227
Writing data for UZ_3238
Writing data for UZ_3239
Writing data for UZ_3240
Writing data for UZ_3241
Writing data for UZ_3243
Writing data for UZ_3246
Writing data for UZ_3247
Writing data for UZ_3248
Writing data for CUP_1001
Writing data for CUP_1002
Writing data 

In [35]:
# save the dataframe

summarized_df.to_csv(f'{search_path}/MNISym_coreg_slope_flipLesion_AtlasSUIT_summarized.tsv', mode = 'w', sep = '\t', index = False, header = True)

In [36]:
summarized_df[summarized_df.subj_id == 'CU_2310']

,image,image_name,frame,region,regionname,volume,atlas,map,space,mean,...,subj_id,ID,Centre,RefT1,age,Gender,isPatient,LesionSide,LesionLocation,handedness
0,1,CU_2310_MNISym_CSF_coreg_reslice_slope_FlipLR....,0,1,Left_I_IV,4567.0,Diedrichsen_2009,atl-Anatom,MNISym,0.000076,...,CU_2310,2310.0,CU,W0,57.0,M,1.0,left,subcortical,2.0
1,1,CU_2310_MNISym_CSF_coreg_reslice_slope_FlipLR....,0,2,Right_I_IV,5524.0,Diedrichsen_2009,atl-Anatom,MNISym,-0.000124,...,CU_2310,2310.0,CU,W0,57.0,M,1.0,left,subcortical,2.0
2,1,CU_2310_MNISym_CSF_coreg_reslice_slope_FlipLR....,0,3,Left_V,5854.0,Diedrichsen_2009,atl-Anatom,MNISym,0.000120,...,CU_2310,2310.0,CU,W0,57.0,M,1.0,left,subcortical,2.0
3,1,CU_2310_MNISym_CSF_coreg_reslice_slope_FlipLR....,0,4,Right_V,5975.0,Diedrichsen_2009,atl-Anatom,MNISym,-0.000030,...,CU_2310,2310.0,CU,W0,57.0,M,1.0,left,subcortical,2.0
4,1,CU_2310_MNISym_CSF_coreg_reslice_slope_FlipLR....,0,5,Left_VI,12898.0,Diedrichsen_2009,atl-Anatom,MNISym,0.000081,...,CU_2310,2310.0,CU,W0,57.0,M,1.0,left,subcortical,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131,4,CU_2310_MNISym_T1_coreg_reslice_slope_FlipLR.n...,0,30,Right_Dentate,2197.0,Diedrichsen_2009,atl-Anatom,MNISym,-7.764727,...,CU_2310,2310.0,CU,W0,57.0,M,1.0,left,subcortical,2.0
132,4,CU_2310_MNISym_T1_coreg_reslice_slope_FlipLR.n...,0,31,Left_Interposed,255.0,Diedrichsen_2009,atl-Anatom,MNISym,-7.806381,...,CU_2310,2310.0,CU,W0,57.0,M,1.0,left,subcortical,2.0
133,4,CU_2310_MNISym_T1_coreg_reslice_slope_FlipLR.n...,0,32,Right_Interposed,277.0,Diedrichsen_2009,atl-Anatom,MNISym,-7.542033,...,CU_2310,2310.0,CU,W0,57.0,M,1.0,left,subcortical,2.0
134,4,CU_2310_MNISym_T1_coreg_reslice_slope_FlipLR.n...,0,33,Left_Fastigial,2.0,Diedrichsen_2009,atl-Anatom,MNISym,-7.409941,...,CU_2310,2310.0,CU,W0,57.0,M,1.0,left,subcortical,2.0


In [37]:
# save the dataframe

summarized_df.to_csv(f'{search_path}/MNISym_coreg_slope_AtlasSUIT_summarized.tsv', mode = 'w', sep = '\t', index = False, header = True)

In [38]:
search_path

'/cifs/diedrichsen/data/smarts_cerebellum/Regression'

In [39]:
len(summarized_df.image_name.unique())

196

In [40]:
49*4

196

In [44]:
base_dir = gl.baseDir
def subj_unique_regression_coreg_logJac():

    """
    Function for local use.

    Function to perform regression on coregistered and normalized (to MNISym template) images

    Input:
        This is just for running regression on the log Jacobian iamges.
    """

    for subj in p_df['subj_id'].unique():

        # find each subject's reference image, and run it through the regression
        refT1 = (p_df.loc[(p_df['subj_id']==subj), 'RefT1'].iloc[0]).strip()
        ref_img = f'{base_dir}/MNISym/full_img_coreg/{subj}/{refT1}/{subj}_{refT1}_T1_to-MNI152NLin2009cSymC_mode-image_log_detJ.nii.gz'

        print(f"Regression on {subj} \n")

        intercept_img, slope_img = regression.perform_regression_week(subj_id = subj,
                            reference_img = ref_img
                            )
        

        # if images exist, save them
        if intercept_img is not None and slope_img is not None:
            results_path = f'{base_dir}/Regression/{subj}'
            results_path = Path(results_path)

            # comment this out if this directory already exists
            #results_path.mkdir(parents = True, exist_ok = True)

            nib.save(intercept_img, f'{results_path}/{subj}_logJac_coreg_reslice_intercept.nii.gz')
            nib.save(slope_img, f'{results_path}/{subj}_logJac_coreg_reslice_slope.nii.gz')


In [45]:
subj_unique_regression_coreg_logJac()

Regression on CU_2310 

Regression on CU_2538 

Regression on CU_2663 

Regression on CU_2697 

Regression on CU_2925 

Regression on JHU_2282 

Regression on JHU_2374 

Regression on JHU_2395 

Regression on JHU_2531 

Regression on JHU_2577 

Regression on JHU_2650 

Regression on JHU_2684 

Regression on JHU_2713 

Regression on JHU_2789 

Regression on JHU_3175 

Regression on JHU_3176 

Regression on UZ_2365 

Regression on UZ_2450 

Regression on UZ_2565 

Regression on UZ_2595 

Regression on UZ_2652 

Regression on UZ_2654 

Regression on UZ_2906 

Regression on UZ_3030 

Regression on UZ_3057 

Regression on UZ_3151 

skipping W4
Regression on UZ_3158 

skipping W52
Regression on UZ_3166 

Regression on UZ_3224 

Regression on UZ_3226 

Regression on UZ_3227 

skipping W4
Regression on UZ_3228 

Regression on UZ_3238 

Regression on UZ_3239 

Regression on UZ_3240 

Regression on UZ_3241 

Regression on UZ_3243 

Regression on UZ_3246 

Regression on UZ_3247 

Regression on UZ